# LAB 08  Support Vector Machines
## Student Information
Name: **Nazakat Ali**  
Student ID: **023-22-0146**  
Section: **Buffer Batch**  
GitHub: **https://github.com/Nizakatkhannk/Machine-Learning-Labs**  
Kaggle: **<your kaggle profile>**  
Dataset: Telco Customer Churn (Kaggle)  
Dataset Source: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

> **How this notebook works:** every code cell prints your *own* numbers, and every "Answers" cell below it is written to
> be read next to those numbers. Run all cells top-to-bottom (`Runtime → Run all`), then read each result and
> adjust any answer sentence that doesn't match what *your* run shows (e.g. which kernel won).

## Part 0 — Recap and Setup

In [ ]:
import os, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve,
                             roc_auc_score, precision_recall_curve, average_precision_score)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

sns.set_style('whitegrid')
RANDOM_STATE = 42     # <-- use the SAME values you used in Labs 3/4/6
TEST_SIZE    = 0.2

def say(text): display(Markdown(text))

def get_metrics(y_true, y_pred):
    return {'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall': recall_score(y_true, y_pred),
            'F1': f1_score(y_true, y_pred)}

In [ ]:
# ---- Load the dataset (3 fallbacks: local file -> kagglehub -> manual upload) ----
CSV = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
if not os.path.exists(CSV):
    try:
        try:
            import kagglehub
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'kagglehub'], check=True)
            import kagglehub
        CSV = os.path.join(kagglehub.dataset_download('blastchar/telco-customer-churn'), CSV)
    except Exception as e:
        print('Auto-download failed:', e, '\nPlease upload the CSV manually.')
        from google.colab import files
        CSV = list(files.upload().keys())[0]

df = pd.read_csv(CSV)
print(df.shape)
df.head()

In [ ]:
# ---- SAME cleaning pipeline as Labs 2-6 -------------------------------------------
# To avoid ANY leakage, I split FIRST (same random_state/test_size/stratify => same rows as earlier labs,
# because the split depends only on y and n), then impute / encode using TRAIN information only.
df = df.drop(columns=['customerID'])
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')   # blanks -> NaN
y = df['Churn'].map({'Yes': 1, 'No': 0})
X_raw = df.drop(columns='Churn')

Xr_train, Xr_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

med = Xr_train['TotalCharges'].median()                                    # train median only
Xr_train = Xr_train.assign(TotalCharges=Xr_train['TotalCharges'].fillna(med))
Xr_test  = Xr_test.assign(TotalCharges=Xr_test['TotalCharges'].fillna(med))

X_train = pd.get_dummies(Xr_train, drop_first=True).astype(float)
X_test  = pd.get_dummies(Xr_test,  drop_first=True).astype(float).reindex(columns=X_train.columns, fill_value=0.0)

print('Train:', X_train.shape, ' Test:', X_test.shape)
print('Churn rate  train: %.3f   test: %.3f' % (y_train.mean(), y_test.mean()))

## Part 1 — Why Scaling Matters for SVM

In [ ]:
print(X_train[['tenure','MonthlyCharges','TotalCharges']].agg(['min','max']).T)
print('\nOther columns are 0/1 dummies. Range of TotalCharges is ~%d times larger.' %
      (X_train['TotalCharges'].max() / 1))

MAX_ITER = 200_000   # safety cap so the UNSCALED model cannot hang the Colab session (applied to both models)

def fit_timed(model, Xtr, ytr):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always')
        t0 = time.time(); model.fit(Xtr, ytr); dt = time.time() - t0
    return dt, sorted({x.category.__name__ for x in w})

# 1) unscaled
svm_raw = SVC(kernel='linear', max_iter=MAX_ITER, random_state=RANDOM_STATE)
t_raw, warn_raw = fit_timed(svm_raw, X_train, y_train)
m_raw = get_metrics(y_test, svm_raw.predict(X_test))

# 2) scaled (fit on TRAIN only)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
svm_sc = SVC(kernel='linear', max_iter=MAX_ITER, random_state=RANDOM_STATE)
t_sc, warn_sc = fit_timed(svm_sc, X_train_s, y_train)
m_sc = get_metrics(y_test, svm_sc.predict(X_test_s))

res1 = pd.DataFrame([{**m_raw, 'Train time (s)': t_raw, 'Warnings': ', '.join(warn_raw) or 'none'},
                     {**m_sc,  'Train time (s)': t_sc,  'Warnings': ', '.join(warn_sc) or 'none'}],
                    index=['Linear SVM - UNSCALED', 'Linear SVM - SCALED'])
display(res1.round(4))

In [ ]:
say(f"""**Your numbers (Task 1.1 a):** unscaled → accuracy **{m_raw['Accuracy']:.3f}**, F1 **{m_raw['F1']:.3f}**,
trained in **{t_raw:.1f}s** (warnings: {', '.join(warn_raw) or 'none'}).
Scaled → accuracy **{m_sc['Accuracy']:.3f}**, F1 **{m_sc['F1']:.3f}**, trained in **{t_sc:.1f}s** (warnings: {', '.join(warn_sc) or 'none'}).
The unscaled model was **{t_raw/max(t_sc,1e-9):.1f}× slower**.""")

### Answers — Part 1
**(a)** See the table and the auto-generated sentence above: the scaled model trains far faster, converges cleanly, and is at least as good on F1/accuracy. The unscaled model is slower and may hit the iteration cap and raise a `ConvergenceWarning` ("Solver terminated early"). If it did not warn on your run, say so honestly — the cap `max_iter=200000` is what I used to stop it hanging.

**(b)** A linear SVM looks for `w·x + b = 0` with the widest margin `2/‖w‖`, and everything — the margin, the slack penalty, the dot products in the optimiser — is computed from raw Euclidean geometry. `TotalCharges` ranges in the thousands while the dummy columns only take values 0/1, so the large-range features dominate the dot products and distances. To use the small-range (but often very informative) features such as `Contract_*` or `InternetService_*`, the optimiser needs very large weights on them, and large weights are exactly what the margin objective `½‖w‖²` penalises. The result is a badly-conditioned optimisation problem: slow convergence, and a boundary that is effectively decided by the big-number columns rather than by the most predictive ones.

**(c)** I use the **scaled** version (`StandardScaler` fit on the training set only, then `transform` on test) for every later SVM, as well as for Logistic Regression and KNN, because all of them are distance/dot-product based. Fitting the scaler on the full data would leak test information into training.

## Part 2 — Kernel Comparison

In [ ]:
rows, kernel_models = [], {}
for k in ['linear', 'rbf', 'poly']:
    m = SVC(kernel=k, random_state=RANDOM_STATE)
    t0 = time.time(); m.fit(X_train_s, y_train); dt = time.time() - t0
    pred = m.predict(X_test_s)
    rows.append({'Kernel': k, **get_metrics(y_test, pred), 'Support vectors': m.support_.shape[0],
                 'SV % of train': 100 * m.support_.shape[0] / len(y_train), 'Train time (s)': dt})
    kernel_models[k] = m
kern = pd.DataFrame(rows).set_index('Kernel')
display(kern.round(4))

best_k = kern['F1'].idxmax()
ConfusionMatrixDisplay.from_predictions(y_test, kernel_models[best_k].predict(X_test_s),
                                        display_labels=['Stay', 'Churn'], cmap='Blues')
plt.title(f'Confusion matrix — best-F1 kernel: {best_k}'); plt.show()

In [ ]:
mostSV = kern['Support vectors'].idxmax()
say(f"""**Your numbers:** best F1 → **{best_k}** ({kern.loc[best_k,'F1']:.3f});
most support vectors → **{mostSV}** ({int(kern.loc[mostSV,'Support vectors'])} SVs, {kern.loc[mostSV,'SV % of train']:.1f}% of the training set).
F1 spread between best and worst kernel = {kern['F1'].max()-kern['F1'].min():.3f}.""")

### Interpretation of the confusion matrix
Read it off the plot above: the diagonal-heavy cells are the many correctly-kept loyal customers, while the bottom row shows how many real churners the default (unweighted) model still misses. Because the data is ~73/27 imbalanced, the default SVM is biased towards predicting "Stay", so false negatives are the visible weakness — this motivates Part 5.

### Answers — Part 2
**(a)** The winner is printed above (`best_k`). On one-hot-heavy tabular data like Telco, the classes are largely separable by a near-linear combination of features (contract type, tenure, internet service, charges), so it is **not** surprising if the **linear kernel is competitive with or equal to RBF** — the theory says RBF is *more flexible*, not *always better*; extra flexibility only pays off when the true boundary is non-linear, and otherwise it mostly adds variance. If poly is worst, that matches theory too: a degree-3 polynomial on ~30 mostly-binary features is easily over-flexible and numerically awkward.

**(b)** The kernel with the highest `Support vectors` count (see table) is learning the most complicated boundary: many points sit on or inside the margin, i.e. the classes overlap a lot and the model has to bend around many individual points. A large SV percentage also means slower prediction (each prediction sums over all SVs) and a higher risk of overfitting.

**(c)** I carry **RBF** forward to tuning, as Part 4 requires, and justify it with the table: its F1 is within the margin of the best kernel (or is the best), it has a tunable `gamma` in addition to `C` (so it can reproduce a near-linear boundary when `gamma` is small *and* bend when needed), and its training time/SV count are acceptable. If your linear kernel clearly won on F1 by a margin larger than the noise (~0.01), say that linear is the simpler choice and that RBF is carried forward only to satisfy the lab's tuning exercise.

## Part 3 — Visualizing the Margin: What Does C Actually Do?

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
Z_train = pca.fit_transform(X_train_s)      # fit PCA on train only
Z_test  = pca.transform(X_test_s)
evr = pca.explained_variance_ratio_
print('Explained variance ratio:', evr.round(4), '-> total = %.1f%%' % (100 * evr.sum()))

C_values = [0.01, 1, 100]
pad = 1.0
xx, yy = np.meshgrid(np.linspace(Z_train[:,0].min()-pad, Z_train[:,0].max()+pad, 250),
                     np.linspace(Z_train[:,1].min()-pad, Z_train[:,1].max()+pad, 250))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
res3 = []
for ax, C in zip(axes, C_values):
    m2 = SVC(kernel='rbf', C=C, gamma='scale', random_state=RANDOM_STATE).fit(Z_train, y_train)
    Zg = m2.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Zg, alpha=0.3, cmap='coolwarm')
    ax.scatter(Z_train[:,0], Z_train[:,1], c=y_train, cmap='coolwarm', s=6, edgecolor='k', linewidth=0.1)
    f_tr = f1_score(y_train, m2.predict(Z_train)); f_te = f1_score(y_test, m2.predict(Z_test))
    res3.append({'C': C, 'Support vectors': m2.support_.shape[0], 'Train F1 (2-D)': f_tr, 'Test F1 (2-D)': f_te,
                 'Train acc': m2.score(Z_train, y_train), 'Test acc': m2.score(Z_test, y_test)})
    ax.set_title(f'RBF SVM, C={C}\n#support vectors = {m2.support_.shape[0]}  |  train F1={f_tr:.2f}, test F1={f_te:.2f}')
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.suptitle(f'Decision boundary in a 2-D PCA projection (PCA keeps only {100*evr.sum():.1f}% of the variance)', y=1.03)
plt.tight_layout(); plt.show()
res3 = pd.DataFrame(res3).set_index('C'); display(res3.round(4))

In [ ]:
say(f"""**Your numbers:** the 2 PCs capture only **{100*evr.sum():.1f}%** of the variance.
Support vectors for C = {C_values[0]}, {C_values[1]}, {C_values[2]}: **{', '.join(str(int(v)) for v in res3['Support vectors'])}**.
Train→test F1 gap per C: {', '.join(f"{c}: {a-b:+.3f}" for c,a,b in zip(res3.index, res3['Train F1 (2-D)'], res3['Test F1 (2-D)']))}.""")

### Interpretation of the plots
Red/blue shading shows the predicted class; the dots are the real training labels. With the smallest C the shaded regions are large and simple, and (because churners are the minority and overlap heavily with non-churners in this projection) the model is willing to absorb many mistakes — it tends to paint almost everything as "Stay". As C grows, the boundary is allowed to carve out small pockets around individual points. Compare these visual impressions with the train-vs-test F1 table.

### Answers — Part 3
**(a)** With small C the boundary is smooth/simple with a wide margin (misclassified training points are cheap, so the model prefers a small `‖w‖`). As C increases the penalty on margin violations grows, so the boundary becomes more flexible and wiggly, with islands around individual points and a narrower margin.

**(b)** See the support-vector counts in the table/plot titles. Theory predicts the number of SVs **falls as C increases** (a wide soft margin has many points inside it, all of which become SVs; a narrow margin has fewer). In this heavily overlapping 2-D projection the count may fall only modestly, because many points stay on the wrong side of the boundary regardless of C — those remain SVs (with α = C). Mention whether your numbers followed the theory.

**(c)** *(Check this against your own plots and the train/test F1 table — the 2-D projection overlaps heavily, so differences can be subtle.)* **C = 100** is normally the overfitting one: boundary has small pockets around individual points and its train F1 exceeds its test F1 by the largest gap in the table. **C = 0.01** is normally the underfitting one: the boundary is nearly flat/one-sided and recall for the churn class is lowest, so it fails even on the training data. C = 1 is the compromise.

**(d)** (i) Two PCs explain only the percentage printed above, so most of the structure the real model uses is thrown away; (ii) PCA is *unsupervised* — it maximises variance, not class separation, so classes that are separable in 30-D can overlap completely in the plot; (iii) the SVM drawn here is *trained on the 2-D projection*, i.e. a different model with different support vectors, `gamma='scale'` and margins, not a slice of the real one; (iv) distances in the projection are distorted versions of real distances, so the apparent margin and the apparent "overfitting" can be misleading.

## Part 4 — Systematic Hyperparameter Search

In [ ]:
param_grid = {'C': [0.1, 1, 10, 100],
              'gamma': [0.0001, 0.001, 0.01, 0.1, 1]}
gs = GridSearchCV(SVC(kernel='rbf', random_state=RANDOM_STATE), param_grid, cv=5, scoring='f1', n_jobs=-1, refit=True)
t0 = time.time(); gs.fit(X_train_s, y_train); t_grid = time.time() - t0
print('Grid search took %.1f s for %d fits' % (t_grid, len(gs.cv_results_['params']) * 5))
print('Best params:', gs.best_params_, ' | best CV F1 = %.4f' % gs.best_score_)

cvr = pd.DataFrame(gs.cv_results_)
heat = cvr.pivot(index='param_C', columns='param_gamma', values='mean_test_score').astype(float)
plt.figure(figsize=(8, 5))
sns.heatmap(heat, annot=True, fmt='.3f', cmap='viridis', cbar_kws={'label': 'mean CV F1'})
plt.xlabel('gamma'); plt.ylabel('C'); plt.title('GridSearchCV — mean 5-fold F1 (RBF SVM)'); plt.show()

In [ ]:
best_svm = gs.best_estimator_
pred_best = best_svm.predict(X_test_s)
m_best = get_metrics(y_test, pred_best)
display(pd.DataFrame([m_best], index=['Tuned RBF SVM (test set)']).round(4))
ConfusionMatrixDisplay.from_predictions(y_test, pred_best, display_labels=['Stay', 'Churn'], cmap='Blues')
plt.title('Tuned RBF SVM — confusion matrix (test)'); plt.show()

near_best = int((heat.values >= heat.values.max() - 0.01).sum())
say(f"""**Your numbers:** best **C = {gs.best_params_['C']}**, **gamma = {gs.best_params_['gamma']}**, CV F1 = **{gs.best_score_:.3f}**.
Test F1 = **{m_best['F1']:.3f}** (gap CV→test = {gs.best_score_-m_best['F1']:+.3f}).
Untuned RBF from Part 2: F1 = {kern.loc['rbf','F1']:.3f} → tuning changed F1 by **{m_best['F1']-kern.loc['rbf','F1']:+.3f}** at a cost of {t_grid:.0f}s of search.
Number of grid cells within 0.01 F1 of the best: **{near_best}** out of {heat.size}.""")

### Interpretation of the heatmap
Bright cells are good (high mean CV F1). Look for (1) a diagonal band: large C with small gamma, or small C with large gamma, tend to give similar boundaries (the two parameters trade off); (2) very large `gamma` with large `C` at the bottom/right is typically dark — that is the overfitting corner where each point carves its own island; (3) very small `gamma` with small `C` is also dark — underfitting.

### Answers — Part 4
**(a)** Best C, gamma and CV F1 are printed above (`gs.best_params_`, `gs.best_score_`).

**(b)** Count how many cells are close to the best (the "near best" number above). If several neighbouring cells are within ~0.01 F1, there is a broad plateau and the model is **not** very sensitive to the exact values — only the order of magnitude matters, which is typical for SVMs on standardised data. A single bright cell surrounded by dark ones would instead mean a fragile optimum.

**(c)** The CV score and test score should agree within roughly 0.01–0.03. A large gap (test ≪ CV) would suggest overfitting of the hyperparameters to the CV folds, a lucky/unrepresentative split, or leakage; a test score much *higher* than CV suggests an easy/unrepresentative test set. Here compare the printed gap.

**(d)** Compare the printed F1 change versus the untuned `rbf` row in Part 2. The default (`C=1, gamma='scale'`) already sits in the plateau, so the improvement is usually small; I judge whether a gain of that size justified the grid-search cost (20 combinations × 5 folds = 100 fits). If the gain is below about 0.01 F1, tuning was **not** worth it purely for F1 — but it did tell us the model is robust, which has value in itself.

## Part 5 — Class Imbalance and the Business Trade-off

In [ ]:
bp = gs.best_params_
svm_def = SVC(kernel='rbf', C=bp['C'], gamma=bp['gamma'], class_weight=None,       random_state=RANDOM_STATE)
svm_bal = SVC(kernel='rbf', C=bp['C'], gamma=bp['gamma'], class_weight='balanced',
              probability=True, random_state=RANDOM_STATE)          # probability=True -> Platt scaling

t0 = time.time(); svm_def.fit(X_train_s, y_train); t_def = time.time() - t0
t0 = time.time(); svm_bal.fit(X_train_s, y_train); t_bal = time.time() - t0

pred_def, pred_bal = svm_def.predict(X_test_s), svm_bal.predict(X_test_s)
m_def, m_bal = get_metrics(y_test, pred_def), get_metrics(y_test, pred_bal)
display(pd.DataFrame([m_def, m_bal], index=['class_weight=None', "class_weight='balanced'"]).round(4))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for a, p, t in zip(ax, [pred_def, pred_bal], ['class_weight=None', "class_weight='balanced'"]):
    ConfusionMatrixDisplay.from_predictions(y_test, p, display_labels=['Stay', 'Churn'], cmap='Blues', ax=a, colorbar=False)
    a.set_title(t)
plt.tight_layout(); plt.show()

In [ ]:
proba = svm_bal.predict_proba(X_test_s)[:, 1]
fpr, tpr, _ = roc_curve(y_test, proba); auc = roc_auc_score(y_test, proba)
prec_c, rec_c, _ = precision_recall_curve(y_test, proba); ap = average_precision_score(y_test, proba)

tn, fp, fn, tp = confusion_matrix(y_test, pred_bal).ravel()
op_fpr, op_tpr = fp / (fp + tn), tp / (tp + fn)       # operating point of predict() (default threshold)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(fpr, tpr, label=f'ROC (AUC = {auc:.3f})'); ax[0].plot([0,1],[0,1],'k--', alpha=.4)
ax[0].scatter([op_fpr], [op_tpr], c='red', s=90, zorder=5, label=f'default threshold (TPR={op_tpr:.2f}, FPR={op_fpr:.2f})')
ax[0].set_xlabel('False positive rate'); ax[0].set_ylabel('True positive rate (recall)'); ax[0].set_title('ROC — balanced SVM'); ax[0].legend()
ax[1].plot(rec_c, prec_c, label=f'PR (AP = {ap:.3f})')
ax[1].scatter([m_bal['Recall']], [m_bal['Precision']], c='red', s=90, zorder=5, label='default threshold')
ax[1].axhline(y_test.mean(), color='k', ls='--', alpha=.4, label='no-skill baseline')
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision'); ax[1].set_title('Precision-Recall — balanced SVM'); ax[1].legend()
plt.tight_layout(); plt.show()

say(f"""**Your numbers (5a):** precision **{m_def['Precision']:.3f} → {m_bal['Precision']:.3f}** ({m_bal['Precision']-m_def['Precision']:+.3f}),
recall **{m_def['Recall']:.3f} → {m_bal['Recall']:.3f}** ({m_bal['Recall']-m_def['Recall']:+.3f}),
F1 {m_def['F1']:.3f} → {m_bal['F1']:.3f}. Missed churners (FN): **{confusion_matrix(y_test, pred_def)[1,0]} → {fn}**;
false alarms (FP): **{confusion_matrix(y_test, pred_def)[0,1]} → {fp}**. ROC-AUC = {auc:.3f}.""")

### Interpretation of the curves
The ROC curve shows how much recall (TPR) you can buy for each unit of false-alarm rate; the area above the diagonal is the model's real ranking skill. The red dot is where the default `predict()` decision lands: with `class_weight='balanced'` it sits much higher (more churners caught) but further right (more loyal customers flagged) than the unweighted model would. On the PR curve, precision falls as recall rises — the whole curve must stay above the no-skill baseline (the churn rate ≈ 27%).

### Answers — Part 5
**(a)** The exact precision/recall movement is printed above. The expected pattern is **recall up sharply, precision down modestly**, because `balanced` multiplies the C penalty of the minority class by `n_samples / (2·n_churn)` ≈ 1.9, so the model is punished much more for missing a churner than for a false alarm.

**(b)** For a telecom company the **false negative is more costly**: acquiring a replacement customer typically costs several times more than retaining one, and a missed churner means lost recurring revenue, whereas a false positive only costs a retention offer (a small discount, a call) sent to someone who would have stayed anyway. This asymmetry means the business should accept lower precision for higher recall — up to the point where retention-campaign budget is exhausted. I therefore **recommend deploying the `class_weight='balanced'` model**, provided the retention team can handle the larger list of flagged customers; if their capacity is very limited, I would instead raise the decision threshold using the PR curve.

**(c)** `SVC` has no native probabilities; its raw output is the signed distance `decision_function`. To get probability-like scores I set **`probability=True`** in the constructor, which makes scikit-learn fit a **Platt-scaling** (sigmoid) calibration on top of the SVM using internal 5-fold cross-validation, and then call `predict_proba`. The cost is a noticeably slower `fit`, and the probabilities can occasionally disagree with `predict()` near the boundary because they're calibrated separately. (Using `decision_function` directly would be a cheaper alternative for ROC/PR curves.)

## Part 6 — Model Showdown: SVM vs. Everything Else

> **Note:** the lab asks for five algorithms; the SVM appears twice (default weights vs `balanced`) because that contrast is the point of Part 5. If you saved exact numbers/hyper-parameters from Labs 3, 4 and 6, replace the settings in the next cell with
> yours (e.g. your Lab 3 `max_depth`, your Lab 4 best RF parameters, your Lab 6 best `k`). The code below re-runs the models
> on **this lab's split** so that the comparison is fair, tuning each one lightly with CV on the training set only.

In [ ]:
results, fit_times = {}, {}

def evaluate(name, model, Xtr, Xte, fit=True, ftime=None):
    if fit:
        t0 = time.time(); model.fit(Xtr, y_train); ftime = time.time() - t0
    results[name] = {**get_metrics(y_test, model.predict(Xte)), 'Fit time (s)': ftime}
    return model

# Decision Tree (Lab 3) — depth chosen by CV
dt_gs = GridSearchCV(DecisionTreeClassifier(random_state=RANDOM_STATE), {'max_depth': [3,4,5,6,8,10], 'min_samples_leaf': [1,5,20]},
                     cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)
evaluate('Decision Tree', DecisionTreeClassifier(random_state=RANDOM_STATE, **dt_gs.best_params_), X_train, X_test)

# Random Forest (Lab 4, tuned)
rf_gs = GridSearchCV(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
                     {'n_estimators': [200, 400], 'max_depth': [6, 10, None], 'min_samples_leaf': [1, 5]},
                     cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)
evaluate('Random Forest (tuned)', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **rf_gs.best_params_), X_train, X_test)

# Logistic Regression (Lab 6) — scaled
evaluate('Logistic Regression', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE), X_train_s, X_test_s)

# KNN (Lab 6) — scaled, k chosen by CV
knn_gs = GridSearchCV(KNeighborsClassifier(), {'n_neighbors': [3,5,7,9,11,15,21,31]}, cv=5, scoring='f1', n_jobs=-1).fit(X_train_s, y_train)
evaluate('KNN', KNeighborsClassifier(**knn_gs.best_params_), X_train_s, X_test_s)

# SVM rows: the tuned model with default weights (Part 4) and the class-weighted model (Part 5)
evaluate('SVM (RBF, tuned)', svm_def, X_train_s, X_test_s, fit=False, ftime=t_def)
evaluate('SVM (RBF, balanced)', svm_bal, X_train_s, X_test_s, fit=False, ftime=t_bal)
svm_name = 'SVM (RBF, balanced)'

print('Chosen params ->  DT:', dt_gs.best_params_, '| RF:', rf_gs.best_params_, '| KNN:', knn_gs.best_params_)
comp = pd.DataFrame(results).T[['Accuracy', 'Precision', 'Recall', 'F1', 'Fit time (s)']]
display(comp.round(4))

In [ ]:
order = comp['F1'].sort_values(ascending=False)
plt.figure(figsize=(10, 4.8))
bars = plt.bar(order.index, order.values, color=sns.color_palette('deep', len(order)))
for b, v in zip(bars, order.values): plt.text(b.get_x() + b.get_width()/2, v + 0.005, f'{v:.3f}', ha='center')
plt.ylabel('F1-score (churn class, test set)'); plt.title('F1-score: five algorithms (SVM shown with and without class weights)')
plt.xticks(rotation=15); plt.ylim(0, order.max() + 0.08); plt.tight_layout(); plt.show()

wF, wR = comp['F1'].idxmax(), comp['Recall'].idxmax()
say(f"""**Your numbers:** best F1 → **{wF}** ({comp.loc[wF,'F1']:.3f}); best recall → **{wR}** ({comp.loc[wR,'Recall']:.3f});
same model? **{'yes' if wF == wR else 'no'}**.
{svm_name} fit time = {comp.loc[svm_name,'Fit time (s)']:.2f}s vs Random Forest {comp.loc['Random Forest (tuned)','Fit time (s)']:.2f}s and Decision Tree {comp.loc['Decision Tree','Fit time (s)']:.3f}s.""")

### Interpretation of the bar chart
The bars compare F1 on the churn class only, so they are not inflated by the majority class the way accuracy is. Differences of ~0.01 between neighbouring bars are within the noise of a single 20 % test split, so treat those as ties; large gaps are meaningful.

### Answers — Part 6
**(a)** The F1 winner and recall winner are printed above (the table has two SVM rows — the tuned default-weight model from Part 4 and the balanced model from Part 5 — so you can see what class weighting buys). The models that don't use class weighting (Decision Tree depth-limited, tuned RF, LR, KNN) all optimise overall accuracy and therefore under-predict churn, which gives them lower recall. The `class_weight='balanced'` SVM usually wins on **recall** and is competitive or best on **F1**; if the two winners differ in your run, say which trade-off each represents.

**(b)** Compare the "Fit time" column with the F1 gain. SVM (especially with `probability=True`) is slower than a tree/forest, and its cost grows between O(n²) and O(n³) with the number of samples, but at ~5,600 rows the absolute cost is a few seconds, which is negligible for a churn model retrained monthly. So the extra cost is **worth it only if** the recall/F1 advantage in the table is meaningful (> ~0.02); if LR is within 0.01 F1, LR would be the more economical choice.

**(c)** I recommend deploying **the balanced RBF SVM**: it gives the highest (or near-highest) recall — which is the business-critical metric because missed churners are expensive — with an F1 that is at least as good as the alternatives, so I am not buying recall with an unacceptable collapse in precision. Cite your own recall, precision and F1 from the table when you write this in your submission, and mention that accuracy alone would have picked a different (less useful) model.

## Part 7 — Final Predictions and Wrap-Up

In [ ]:
final_model = svm_bal                                   # balanced RBF SVM (has probability=True)
final_pred  = final_model.predict(X_test_s)
final_proba = final_model.predict_proba(X_test_s)[:, 1]
out = pd.DataFrame({'true_label': y_test.values,
                    'predicted_label': final_pred,
                    'churn_probability': final_proba.round(4),
                    'decision_score': final_model.decision_function(X_test_s).round(4)},
                   index=y_test.index)
out.to_csv('svm_predictions.csv', index_label='row_id')
print('Saved svm_predictions.csv with', len(out), 'rows')
display(out.head(10))
try:
    from google.colab import files; files.download('svm_predictions.csv')
except Exception:
    pass

In [ ]:
mf = get_metrics(y_test, final_pred)
sens = 'low' if near_best >= 4 else ('moderate' if near_best >= 2 else 'fairly high')
say(f"""### Executive Summary
I recommend deploying an **RBF-kernel SVM with `class_weight='balanced'`** (C = {bp['C']}, gamma = {bp['gamma']}), trained on standardised features.
On the held-out test set it reaches **recall {mf['Recall']:.2f}, precision {mf['Precision']:.2f}, F1 {mf['F1']:.2f}, accuracy {mf['Accuracy']:.2f}** (ROC-AUC {auc:.2f}).
Compared with the unweighted SVM, recall moved from {m_def['Recall']:.2f} to {m_bal['Recall']:.2f} while precision moved from {m_def['Precision']:.2f} to {m_bal['Precision']:.2f};
I accepted that precision loss deliberately, since missing a churner costs the company far more than sending a retention offer to a customer who would have stayed.
Scaling was essential (the unscaled linear SVM was {t_raw/max(t_sc,1e-9):.1f}x slower), and the C×gamma heatmap showed {sens} sensitivity to hyper-parameters ({near_best} of {heat.size} grid cells within 0.01 F1 of the best).
Among the five algorithms, {wF} had the best F1 ({comp.loc[wF,'F1']:.2f}) and {wR} the best recall ({comp.loc[wR,'Recall']:.2f}).
**Limitation:** all results come from a single 80/20 split of one dataset, so differences of about 0.01 are within noise, and the model has not been checked for probability calibration or tested on future (out-of-time) customers.""")

### Final Reflection
**(a) Most surprising result:** *(write your own candidates, depending on your run: how large the slow-down/convergence problem was without scaling; how close linear and RBF kernels were; how flat the C×gamma heatmap was; how much recall jumped with `class_weight='balanced'`; or that logistic regression stayed competitive with the SVM.)*

**(b) With one more hour:** (1) feature engineering e.g. tenure buckets, number of add-on services, charge-per-month ratios; (2) threshold tuning on the PR curve using an explicit cost ratio (e.g. FN = 5× FP) instead of relying on `class_weight`; (3) SMOTE or under-sampling inside a cross-validation pipeline and compare with `class_weight`; (4) a repeated stratified k-fold evaluation to put confidence intervals on the F1 differences; (5) a supervised projection (LDA) for Part 3's plot instead of PCA.